In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import os
import requests

# ==========================
# 저장 경로
# ==========================
SAVE_PATH = "cdesk"
os.makedirs(SAVE_PATH, exist_ok=True)

# ==========================
# 책상 검색 키워드
# ==========================
keywords = [
    "neat home desk photo -render -3d -illustration -cartoon -sketch -cgi",
    "minimalist desk photo -render -3d -illustration -cartoon -sketch -cgi",
    "clean study desk photo -render -3d -illustration -cartoon -sketch -cgi"
]

# ==========================
# 크롬 실행
# ==========================
driver = wb.Chrome()
count = 0
downloaded_urls = set()

# ==========================
# 크롤링 시작
# ==========================
for keyword in keywords:
    print(f"\n===== {keyword} 크롤링 시작 =====")
    driver.get("https://images.google.com/")
    time.sleep(1)

    search = driver.find_element(By.NAME, "q")
    search.clear()
    search.send_keys(keyword)
    search.send_keys(Keys.ENTER)
    time.sleep(2)

    body = driver.find_element(By.TAG_NAME, "body")

    # 스크롤
    for _ in range(40):
        body.send_keys(Keys.END)
        time.sleep(0.3)

    # 이미지 태그 수집
    imgs = driver.find_elements(By.TAG_NAME, "img")
    for img in imgs:
        img_url = img.get_attribute("src")
        if img_url is None or not img_url.startswith("http"):
            continue
        if img_url in downloaded_urls:
            continue
        try:
            headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            response = requests.get(img_url, headers=headers, timeout=5)

            # 작은 이미지 제거 (5000B 유지)
            if len(response.content) < 5000:
                continue
            file_path = os.path.join(SAVE_PATH, f"clean_desk_{count}.jpg")
            with open(file_path, "wb") as f:
                f.write(response.content)
            downloaded_urls.add(img_url)
            print(f"{file_path} 저장 완료")
            count += 1
            time.sleep(0.2)
        except Exception as e:
            print("다운로드 실패:", e)
            continue

driver.quit()
print("\n===== dirty desk 이미지 크롤링 완료 =====")
print(f"총 다운로드: {count} 장")